In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoImageProcessor
from typing import List, Dict, Optional
import PIL.Image

class DINOv3(nn.Module):
    """
    Configurable DINOv3 model with flexible return options.
    Supports returning hidden states and feature maps in various formats.
    """
    
    def __init__(self, config):
        """
        Initialize DINOv3 model with configuration.
        
        Args:
            config: dict containing:
                - model_name: str, DINOv3 model name (default: "facebook/dinov3-vitb16-pretrain-lvd1689m")
                - image_size: int, input image size (default: 896). The input image will be preprocessed to have this size (square)
                - freeze_backbone: bool, whether to freeze DINOv3 parameters (default: True)
                - return_last_hidden_state: bool, return last hidden state (default: True)
                - return_all_hidden_states: bool, return all hidden states (default: False)
                - return_selected_layers: list[int], specific layer indices to return (default: None)
                - return_as_feature_maps: bool, reshape patch tokens to spatial format (default: False)
                - return_cls_token: bool, return CLS token (default: False)
                - return_register_tokens: bool, return register tokens (default: False)
        """
        super().__init__()
        
        # Configuration with defaults
        self.config = {
            'model_name': "facebook/dinov3-vitb16-pretrain-lvd1689m",
            'image_size': 896,
            'freeze_backbone': True,
            'return_last_hidden_state': True,
            'return_all_hidden_states': False,
            'return_selected_layers': None,
            'return_as_feature_maps': False,
            'return_cls_token': False,
            'return_register_tokens': False,
            **config  # Override defaults with user config
        }
        
        # DINOv3 backbone
        self.dinov3 = AutoModel.from_pretrained(self.config['model_name'])
        self.processor = AutoImageProcessor.from_pretrained(self.config['model_name'])
        self.processor.size = {'height': self.config['image_size'], 'width': self.config['image_size']}
        
        # Freeze parameters if requested
        if self.config['freeze_backbone']:
            for param in self.dinov3.parameters():
                param.requires_grad = False
        
        # Model properties
        self.feature_dim = self.dinov3.config.hidden_size  # 768 for ViT-B/16
        self.patch_size = self.dinov3.config.patch_size    # 16 for DINOv3
        self.dinov3.config.image_size = self.config['image_size']
    
    def get_patch_spatial_dims(self, input_height, input_width):
        """Calculate spatial dimensions of patch features based on input size."""
        patch_h = input_height // self.patch_size
        patch_w = input_width // self.patch_size
        return patch_h, patch_w
    
    def extract_cls_token(self, hidden_states):
        """
        Extract CLS token from hidden states.
        
        Args:
            hidden_states: [B, N_tokens, feature_dim] - Raw hidden states from DINOv3
        
        Returns:
            [B, feature_dim] - CLS token features
        """
        return hidden_states[:, 0]  # CLS token is at index 0
    
    def extract_register_tokens(self, hidden_states):
        """
        Extract register tokens from hidden states.
        
        Args:
            hidden_states: [B, N_tokens, feature_dim] - Raw hidden states from DINOv3
        
        Returns:
            [B, 4, feature_dim] - Register token features (4 register tokens)
        """
        return hidden_states[:, 1:5]  # Register tokens are at indices 1-4
    
    def tokens_to_feature_maps(self, hidden_states, batch_size, patch_h, patch_w):
        """
        Convert patch tokens to spatial feature maps.
        
        Args:
            hidden_states: [B, N_tokens, feature_dim] - Raw hidden states from DINOv3
            batch_size: int - Batch size
            patch_h, patch_w: int - Spatial dimensions of patches
        
        Returns:
            [B, feature_dim, patch_h, patch_w] - Spatial feature maps
        """
        # Remove CLS token (index 0) and register tokens (indices 1-4)
        # DINOv3 has 1 CLS token + 4 register tokens + patch tokens
        patch_tokens = hidden_states[:, 5:]  # [B, N_patches, feature_dim]
        
        # Reshape to spatial format
        patch_tokens = patch_tokens.transpose(1, 2)  # [B, feature_dim, N_patches]
        feature_maps = patch_tokens.view(batch_size, self.feature_dim, patch_h, patch_w)
        
        return feature_maps
    
    def forward(self, rgb_image):
        """
        Forward pass with configurable return options.
        
        Args:
            rgb_image: [B, 3, H, W] - Input RGB images (should be preprocessed for DINOv3)
                      Can be any size as long as H and W are divisible by patch_size
        
        Returns:
            dict containing requested outputs based on config:
                - 'last_hidden_state': [B, N_tokens, feature_dim] or [B, feature_dim, patch_h, patch_w]
                - 'cls_token': [B, feature_dim] - CLS token features
                - 'register_tokens': [B, 4, feature_dim] - Register token features (4 tokens)
                - 'all_hidden_states': List of [B, N_tokens, feature_dim] or [B, feature_dim, patch_h, patch_w]
                - 'selected_hidden_states': List of selected layer outputs
        """
        batch_size, _, input_h, input_w = rgb_image.shape
        
        # Ensure input dimensions are compatible with patch size
        assert input_h % self.patch_size == 0, f"Height {input_h} must be divisible by patch size {self.patch_size}"
        assert input_w % self.patch_size == 0, f"Width {input_w} must be divisible by patch size {self.patch_size}"
        
        # Calculate patch spatial dimensions
        patch_h, patch_w = self.get_patch_spatial_dims(input_h, input_w)
        
        # Get DINOv3 outputs
        need_all_hidden_states = (self.config['return_all_hidden_states'] or 
                                 self.config['return_selected_layers'] is not None)
        
        outputs = self.dinov3(rgb_image, output_hidden_states=need_all_hidden_states)
        
        # Prepare return dictionary
        result = {}
        
        # Return last hidden state
        if self.config['return_last_hidden_state']:
            last_hidden = outputs.last_hidden_state  # [B, N_tokens, feature_dim]
            if self.config['return_as_feature_maps']:
                last_hidden = self.tokens_to_feature_maps(last_hidden, batch_size, patch_h, patch_w)
            result['last_hidden_state'] = last_hidden
        
        # Return CLS token
        if self.config['return_cls_token']:
            cls_token = self.extract_cls_token(outputs.last_hidden_state)  # [B, feature_dim]
            result['cls_token'] = cls_token
        
        # Return register tokens
        if self.config['return_register_tokens']:
            register_tokens = self.extract_register_tokens(outputs.last_hidden_state)  # [B, 4, feature_dim]
            result['register_tokens'] = register_tokens
        
        # Return all hidden states
        if self.config['return_all_hidden_states']:
            all_hidden = outputs.hidden_states  # Tuple of [B, N_tokens, feature_dim]
            if self.config['return_as_feature_maps']:
                all_hidden = [self.tokens_to_feature_maps(h, batch_size, patch_h, patch_w) 
                             for h in all_hidden]
            result['all_hidden_states'] = all_hidden
        
        # Return selected hidden states
        if self.config['return_selected_layers'] is not None:
            selected_layers = self.config['return_selected_layers']
            all_hidden = outputs.hidden_states
            selected_hidden = [all_hidden[i] for i in selected_layers]
            if self.config['return_as_feature_maps']:
                selected_hidden = [self.tokens_to_feature_maps(h, batch_size, patch_h, patch_w) 
                                  for h in selected_hidden]
            result['selected_hidden_states'] = selected_hidden
        
        return result
    
    def preprocess_image(self, image_tensor):
        """
        Preprocess image for DINOv3 using the proper processor.
        
        Args:
            image_tensor: [B, 3, H, W] - Raw image tensor with values in [0, 1]
        
        Returns:
            [B, 3, H, W] - Preprocessed tensor ready for DINOv3
        """
        
        # Vectorized conversion to [0, 255] range and permute dimensions
        # [B, 3, H, W] -> [B, H, W, 3]
        img_batch = (image_tensor * 255).byte().permute(0, 2, 3, 1)  # [B, H, W, 3]
        
        # Convert entire batch to numpy for PIL processing
        img_numpy = img_batch.cpu().numpy()  # [B, H, W, 3]
        
        # Create PIL images from the entire batch
        pil_images = [PIL.Image.fromarray(img_numpy[i], mode='RGB') for i in range(img_numpy.shape[0])]
        
        # Process entire batch at once using the processor
        processed = self.processor(images=pil_images, return_tensors="pt")
        processed_batch = processed["pixel_values"]  # [B, 3, H, W]
        
        return processed_batch.to(image_tensor.device)

In [ ]:
from rich import print
# Create a sample input batch
batch_size = 2
input_height, input_width = 999, 999  # Must be divisible by patch_size (16)
sample_input = torch.randn(batch_size, 3, input_height, input_width).cuda()


print(f"Input shape: {sample_input.shape}")
print("=" * 80)

# Test 1: Default configuration - only last hidden state as tokens
print("Test 1: Default configuration (last hidden state as tokens)")
config1 = {}
model1 = DINOv3(config1).cuda()
sample_input_preprocessed = model1.preprocess_image(sample_input) 
with torch.no_grad():
    output1 = model1(sample_input_preprocessed)
for key, value in output1.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
    elif isinstance(value, list):
        print(f"  {key}: List of {len(value)} tensors, shapes: {[v.shape for v in value]}")
print()

# Test 2: Last hidden state as feature maps
print("Test 2: Last hidden state as feature maps")
config2 = {
    'return_as_feature_maps': True
}
model2 = DINOv3(config2).cuda()
sample_input_preprocessed = model2.preprocess_image(sample_input) 
with torch.no_grad():
    output2 = model2(sample_input_preprocessed)
for key, value in output2.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
    elif isinstance(value, list):
        print(f"  {key}: List of {len(value)} tensors, shapes: {[v.shape for v in value]}")
print()

# Test 3: CLS token and register tokens
print("Test 3: CLS token and register tokens")
config3 = {
    'return_cls_token': True,
    'return_register_tokens': True
}
model3 = DINOv3(config3).cuda()
sample_input_preprocessed = model3.preprocess_image(sample_input) 
with torch.no_grad():
    output3 = model3(sample_input_preprocessed)
for key, value in output3.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
    elif isinstance(value, list):
        print(f"  {key}: List of {len(value)} tensors, shapes: {[v.shape for v in value]}")
print()

# Test 4: All hidden states as tokens
print("Test 4: All hidden states as tokens")
config4 = {
    'return_all_hidden_states': True,
    'return_last_hidden_state': False
}
model4 = DINOv3(config4).cuda()
sample_input_preprocessed = model4.preprocess_image(sample_input) 
with torch.no_grad():
    output4 = model4(sample_input_preprocessed)
for key, value in output4.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
    elif isinstance(value, list):
        print(f"  {key}: List of {len(value)} tensors, shapes: {[v.shape for v in value]}")
print()

# Test 5: All hidden states as feature maps
print("Test 5: All hidden states as feature maps")
config5 = {
    'return_all_hidden_states': True,
    'return_as_feature_maps': True,
    'return_last_hidden_state': False
}
model5 = DINOv3(config5).cuda()
sample_input_preprocessed = model5.preprocess_image(sample_input) 
with torch.no_grad():
    output5 = model5(sample_input_preprocessed)
for key, value in output5.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
    elif isinstance(value, list):
        print(f"  {key}: List of {len(value)} tensors, shapes: {[v.shape for v in value]}")
print()

# Test 6: Selected layers as tokens
print("Test 6: Selected layers as tokens (layers 6, 9, 11)")
config6 = {
    'return_selected_layers': [6, 9, 11],
    'return_last_hidden_state': False
}
model6 = DINOv3(config6).cuda()
sample_input_preprocessed = model6.preprocess_image(sample_input) 
with torch.no_grad():
    output6 = model6(sample_input_preprocessed)
for key, value in output6.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
    elif isinstance(value, list):
        print(f"  {key}: List of {len(value)} tensors, shapes: {[v.shape for v in value]}")
print()

# Test 7: Selected layers as feature maps
print("Test 7: Selected layers as feature maps (layers 3, 6, 9)")
config7 = {
    'return_selected_layers': [3, 6, 9],
    'return_as_feature_maps': True,
    'return_last_hidden_state': False
}
model7 = DINOv3(config7).cuda()
sample_input_preprocessed = model7.preprocess_image(sample_input) 
with torch.no_grad():
    output7 = model7(sample_input_preprocessed)
for key, value in output7.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
    elif isinstance(value, list):
        print(f"  {key}: List of {len(value)} tensors, shapes: {[v.shape for v in value]}")
print()

# Test 8: Everything combined - maximum output
print("Test 8: Everything combined (maximum output)")
config8 = {
    'return_last_hidden_state': True,
    'return_all_hidden_states': True,
    'return_selected_layers': [0, 6, 11],
    'return_as_feature_maps': True,
    'return_cls_token': True,
    'return_register_tokens': True
}
model8 = DINOv3(config8).cuda()
sample_input_preprocessed = model8.preprocess_image(sample_input) 
with torch.no_grad():
    output8 = model8(sample_input_preprocessed)
for key, value in output8.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
    elif isinstance(value, list):
        print(f"  {key}: List of {len(value)} tensors, shapes: {[v.shape for v in value]}")
print()

# Test 9: Different image size (smaller)
print("Test 9: Different image size (448x448)")
smaller_input = torch.randn(batch_size, 3, 448, 448).cuda()
config9 = {
    'image_size': 448,
    'return_as_feature_maps': True,
    'return_cls_token': True
}
model9 = DINOv3(config9).cuda()
smaller_input_preprocessed = model9.preprocess_image(smaller_input) 
with torch.no_grad():
    output9 = model9(smaller_input_preprocessed)
for key, value in output9.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
    elif isinstance(value, list):
        print(f"  {key}: List of {len(value)} tensors, shapes: {[v.shape for v in value]}")
print()

# Test 10: Non-square image (rectangular)
print("Test 10: Non-square image (512x768)")
rect_input = torch.randn(batch_size, 3, 512, 768).cuda()
config10 = {
    'return_as_feature_maps': True,
    'return_cls_token': True,
    'return_register_tokens': True
}
model10 = DINOv3(config10).cuda()
rect_input_preprocessed = model10.preprocess_image(rect_input) 
with torch.no_grad():
    output10 = model10(rect_input_preprocessed)
for key, value in output10.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
    elif isinstance(value, list):
        print(f"  {key}: List of {len(value)} tensors, shapes: {[v.shape for v in value]}")

print("=" * 80)
print("Summary of possible output keys and their tensor shapes:")
print("- last_hidden_state: [B, N_tokens, 768] or [B, 768, H_patches, W_patches]")
print("- cls_token: [B, 768]")
print("- register_tokens: [B, 4, 768]")
print("- all_hidden_states: List of 12 tensors, each [B, N_tokens, 768] or [B, 768, H_patches, W_patches]")
print("- selected_hidden_states: List of selected tensors, each [B, N_tokens, 768] or [B, 768, H_patches, W_patches]")
print("\nWhere:")
print("- B = batch_size")
print("- N_tokens = 1 (CLS) + 4 (register) + H_patches * W_patches (patches)")
print("- H_patches = input_height // 16")
print("- W_patches = input_width // 16")
print("- 768 = feature_dim for ViT-B/16 (384 for ViT-S/16)")



Input shape: torch.Size([2, 3, 999, 999])

================================================================================

Test 1: Default configuration (last hidden state as tokens)

last_hidden_state: torch.Size([2, 3141, 768])

Test 2: Last hidden state as feature maps

...

- N_tokens = 1 (CLS) + 4 (register) + H_patches * W_patches (patches)

- H_patches = input_height // 16

- W_patches = input_width // 16

- 768 = feature_dim for ViT-B/16 (384 for ViT-S/16)

: 